In [11]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from time import sleep

In [12]:
limit = 100
records = []
base_url = "https://books.toscrape.com/"
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; StudentDataCollector/1.0)"})

for page in range(1, 6):
    url = urljoin(base_url, f"catalogue/page-{page}.html")
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.select("article.product_pod")
    if not books:
        break
    for book in books:
        title_tag = book.select_one("h3 a")
        price_tag = book.select_one("p.price_color")
        rating_tag = book.select_one("p.star-rating")
        stock_tag = book.select_one("p.instock.availability")
        records.append({
            "title": title_tag.get("title", "").strip(),
            "price_gbp": float(price_tag.get_text(strip=True).replace("£", "")),
            "rating": {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}.get(rating_tag.get("class", [""])[-1]),
            "availability": stock_tag.get_text(" ", strip=True),
            "url": urljoin(url, title_tag["href"])
        })
        if len(records) >= limit:
            break
    if len(records) >= limit:
        break
    sleep(1)

df = pd.DataFrame(records)
print(f"{len(df)} records")
display(df.head())

100 records


,title,price_gbp,rating,availability,url
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [13]:
print(df.info())
display(df.describe(include="all"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         100 non-null    object 
 1   price_gbp     100 non-null    float64
 2   rating        100 non-null    int64  
 3   availability  100 non-null    object 
 4   url           100 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 4.0+ KB
None


,title,price_gbp,rating,availability,url
count,100,100.000000,100.000000,100,100
unique,100,NaN,NaN,1,100
top,A Light in the Attic,NaN,NaN,In stock,https://books.toscrape.com/catalogue/a-light-i...
freq,1,NaN,NaN,100,1
mean,NaN,34.560700,2.930000,NaN,NaN
std,NaN,14.638531,1.423149,NaN,NaN
min,NaN,10.160000,1.000000,NaN,NaN
25%,NaN,19.897500,2.000000,NaN,NaN
50%,NaN,34.775000,3.000000,NaN,NaN
75%,NaN,47.967500,4.000000,NaN,NaN


In [15]:
df.to_csv("books_100.csv", index=False)